<a href="https://colab.research.google.com/github/DianaBarradasSanchez/Simulaci-n-ll/blob/main/Buenas_cadenas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introducción

Vamos a implementar una **Cadena de Márkov** definida como una caminata aleatoria sobre un grafo de **secuencias binarias válidas** (secuencias que no contienen unos adyacentes, $11$).

### Reglas del Modelo:
Dada una secuencia válida de longitud $m$:
1. Se selecciona de manera uniforme al azar un componente $i \in \{1, 2, \dots, m\}$.
2. **Si el componente es $1$:** Se cambia a $0$. La nueva secuencia siempre es válida.
3. **Si el componente es $0$:** Se intenta cambiar a $1$.
   - Si la secuencia resultante es válida, la caminata se desplaza al nuevo estado.
   - Si la secuencia resultante es inválida (crea un $11$), la caminata permanece en el estado actual (se genera un bucle o *loop*).

### Propiedades:
* **Grafo Ponderado:** Dos secuencias válidas con distancia de Hamming igual a $1$ se unen por una arista de peso $1$.
* **Bucles:** El peso del bucle en un estado es igual a la cantidad de intentos de cambio que generan secuencias inválidas.
* **Suma de Pesos:** La suma de los pesos de las aristas incidentes en cualquier vértice es siempre igual a $m$.
* **Distribución Estacionaria:** Dado que todos los vértices tienen el mismo peso total $m$, la distribución estacionaria $\pi$ es **uniforme**:
$$\pi_i = \frac{1}{|V|}, \quad \forall i \in V$$

In [1]:
# FUNCIONES DE VERIFICACIÓN Y GENERACIÓN DE ESTADOS


def es_secuencia_valida(seq: str) -> bool:
    """
    Verifica si una secuencia binaria no contiene unos adyacentes ('11').

    Parámetros:
        seq (str): Cadena binaria a evaluar.

    Retorna:
        bool: True si es válida, False si contiene '11'.
    """
    for i in range(len(seq) - 1):
        if seq[i] == '1' and seq[i+1] == '1':
            return False
    return True


def generar_secuencias_validas(m: int) -> list:
    """
    Genera el conjunto de vértices V del grafo, correspondiente a todas
    las secuencias binarias de longitud m sin '1's adyacentes.

    Parámetros:
        m (int): Longitud de las secuencias binarias.

    Retorna:
        list: Lista de cadenas binarias válidas.
    """
    secuencias_validas = []
    # Evaluamos todas las 2^m combinaciones binarias posibles
    for i in range(2**m):
        seq = format(i, f'0{m}b')
        if es_secuencia_valida(seq):
            secuencias_validas.append(seq)
    return secuencias_validas

# 2. Construcción de la Matriz de Transición $P$

En esta sección construimos la matriz de transición estocástica $P$, donde el elemento $P_{i,j}$ representa la probabilidad de pasar del estado $i$ al estado $j$ en un solo paso:

$$P_{i,j} = \frac{W(i, j)}{m}$$

Donde $W(i, j)$ es el peso de la arista entre la secuencia $i$ y la secuencia $j$.

In [2]:
# CONSTRUCCIÓN DE LA MATRIZ DE TRANSICIÓN P

import numpy as np

def construir_matriz_transicion(m: int):
    """
    Parámetros:
        m (int): Longitud de las secuencias binarias.

    Retorna:
        tuple: (lista de secuencias, matriz de transición P)
    """
    secuencias = generar_secuencias_validas(m)
    num_estados = len(secuencias)

    # Mapeo de secuencia -> índice de matriz
    mapa_estados = {seq: idx for idx, seq in enumerate(secuencias)}

    # Matriz estocástica P de dimensión |V| x |V|
    P = np.zeros((num_estados, num_estados))

    for i, seq in enumerate(secuencias):
        # Diccionario para acumular los pesos incidentes
        pesos_transicion = {s: 0 for s in secuencias}

        # Probar cambiar cada una de las m posiciones
        for pos in range(m):
            list_seq = list(seq)
            bit_actual = list_seq[pos]

            # Invertir el bit en la posición 'pos'
            list_seq[pos] = '0' if bit_actual == '1' else '1'
            nueva_seq = "".join(list_seq)

            if bit_actual == '1':
                # Cambiar 1 -> 0 siempre genera una secuencia válida
                pesos_transicion[nueva_seq] += 1
            else:
                # Cambiar 0 -> 1 solo si la nueva secuencia es válida
                if es_secuencia_valida(nueva_seq):
                    pesos_transicion[nueva_seq] += 1
                else:
                    # Si es inválida, se incrementa el peso del bucle (permanecer en seq)
                    pesos_transicion[seq] += 1

        # Normalización por m para obtener las probabilidades
        for destino, peso in pesos_transicion.items():
            j = mapa_estados[destino]
            P[i, j] = peso / m

    return secuencias, P

# 3. Simulación

A continuación, ejecutamos el modelo para $m = 4$.

Verificaremos que la matriz de transición sea válida y que la **distribución estacionaria** resultante sea perfectamente uniforme:

$$\pi = \left[ \frac{1}{|V|}, \frac{1}{|V|}, \dots, \frac{1}{|V|} \right]$$

In [3]:
# EJECUCIÓN Y COMPROBACIÓN DE DISTRIBUCIÓN ESTACIONARIA


# Definir la dimensión
m = 4

# 1. Generar estados y matriz P
secuencias, P = construir_matriz_transicion(m)

print(f"=== ESTADOS ENCONTRADOS (|V| = {len(secuencias)}) ===")
for idx, seq in enumerate(secuencias):
    print(f"Estado {idx}: {seq}")

print("\n=== MATRIZ DE TRANSICIÓN P ===")
print(np.round(P, 3))

# 2. Cálculo Teórico de la Distribución Estacionaria usando Autovectores
# Se busca resolver: pi * P = pi  =>  P^T * pi^T = pi^T
valores_propios, vectores_propios = np.linalg.eig(P.T)

# Seleccionar el autovector correspondiente al autovalor lambda = 1
idx_lambda_1 = np.argmin(np.abs(valores_propios - 1.0))
pi = np.real(vectores_propios[:, idx_lambda_1])

# Normalizar para que la suma de probabilidades sea 1
pi = pi / np.sum(pi)

print("\n=== DISTRIBUCIÓN ESTACIONARIA CALCULADA (pi) ===")
for idx, seq in enumerate(secuencias):
    print(f"P({seq}) = {pi[idx]:.4f}")

# 3. Comprobación de Uniformidad
es_uniforme = np.allclose(pi, 1.0 / len(secuencias))
print(f"\n¿La distribución estacionaria es perfectamente uniforme?: {es_uniforme}")

=== ESTADOS ENCONTRADOS (|V| = 8) ===
Estado 0: 0000
Estado 1: 0001
Estado 2: 0010
Estado 3: 0100
Estado 4: 0101
Estado 5: 1000
Estado 6: 1001
Estado 7: 1010

=== MATRIZ DE TRANSICIÓN P ===
[[0.   0.25 0.25 0.25 0.   0.25 0.   0.  ]
 [0.25 0.25 0.   0.   0.25 0.   0.25 0.  ]
 [0.25 0.   0.5  0.   0.   0.   0.   0.25]
 [0.25 0.   0.   0.5  0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.25 0.5  0.   0.   0.  ]
 [0.25 0.   0.   0.   0.   0.25 0.25 0.25]
 [0.   0.25 0.   0.   0.   0.25 0.5  0.  ]
 [0.   0.   0.25 0.   0.   0.25 0.   0.5 ]]

=== DISTRIBUCIÓN ESTACIONARIA CALCULADA (pi) ===
P(0000) = 0.1250
P(0001) = 0.1250
P(0010) = 0.1250
P(0100) = 0.1250
P(0101) = 0.1250
P(1000) = 0.1250
P(1001) = 0.1250
P(1010) = 0.1250

¿La distribución estacionaria es perfectamente uniforme?: True
